In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import graphinglib as gl
import pandas as pd
import pyregion
import palettable
import pvextractor
from astropy.constants import c as speed_of_light

from src.hdu.map import Map
from src.hdu.cube import Cube
from src.tools.plotting import *
from src.config import REDSHIFT

In [ ]:
px_to_pc = 20.6/2  # pc/px at the high resolution

AGN_pos = gl.Point(56.368, 51.369, marker_style="x", face_color="red", marker_size=50)

## PV diagram

In [ ]:
cube = Cube.load("data/letter/ngc4696_v7_choiclip1_1a_marquis1-conv_2.5_paa_contsub_1.fits", hdu_index=5)
hm = cube[590, :, :].data.plot.copy_with(color_map_range=(-0.01, 0.04), show_color_bar=False)
data, wavelengths = cube.data, cube.header.wavelengths / (1 + REDSHIFT)

noise_map = np.nanstd(data[615:650, :, :], axis=0)
noise = np.nanmean(
    Map(noise_map, header=cube.header.celestial).get_masked_region(pyregion.open("data/letter/confidence_region.reg")).data
)
cmap_levels = np.arange(3 * noise, 16 * noise, noise)

aperture_colors = palettable.cartocolors.qualitative.Vivid_3.mpl_colors
paths = pvextractor.paths_from_regfile("data/letter/CNDFilament_v2.reg")
apertures_df = pd.DataFrame(columns=["points", "width", "color"], data=list(zip(
    [path.get_xy(cube.header.wcs) for path in paths],
    [4, 3, 3],
    aperture_colors,
)))

x_tick_spacing_pc = 100  # pc
y_tick_spacing = 200  # km/s

cropped_data = data[575:610, :, :]
rest_wavelength = 1.87561e-6

# Calculate velocities for this line
wavelengths_slice = wavelengths[575:610]
velocities = (wavelengths_slice - rest_wavelength) / rest_wavelength * speed_of_light.to("kilometer/s").value

velocity_tick_values = np.arange(
    np.ceil(np.min(velocities) / y_tick_spacing) * y_tick_spacing,
    np.max(velocities),
    y_tick_spacing
)
# Map velocity values to pixel indices
velocity_tick_positions = np.interp(velocity_tick_values, velocities, np.arange(len(velocities)))

cumulative_length_pc = 0  # Track cumulative length in parsecs
aperture_lengths_px = []
figs = []
apertures = []

for aperture in apertures_df.itertuples():
    aperture_poly, bin_polys, aperture_arrow, pv_hm, pv_cont_filled, pv_cont_lines = make_pv_diagram(
        data_cube=cropped_data,
        wcs=cube.header.wcs,
        aperture=aperture.points,
        width=aperture.width,
        spacing=1,
        contour_levels=cmap_levels,
    )

    apertures.extend([
        *[poly.copy_with(edge_color=aperture.color) for poly in [aperture_poly, *bin_polys]],
        aperture_arrow.copy_with(color=aperture.color)
    ])

    # Individual figures
    fig = gl.SmartFigure(elements=[pv_cont_filled, pv_cont_lines], reference_labels=False)
    fig.set_visual_params(axes_edge_color=aperture.color, axes_line_width=2)
    fig.set_tick_params(which="both", draw_left_labels=False, draw_top_ticks=True, draw_right_ticks=True)
    figs.append(fig)

    # Scaling
    aperture_points = np.array(aperture.points)
    current_aperture_length_px = np.linalg.norm(aperture_points[1] - aperture_points[0])
    current_aperture_length_pc = current_aperture_length_px * px_to_pc

    # Calculate where ticks should appear in parsec space (x-axis)
    tick_positions_pc = np.arange(
        np.ceil(cumulative_length_pc / x_tick_spacing_pc) * x_tick_spacing_pc,
        cumulative_length_pc + current_aperture_length_pc,
        x_tick_spacing_pc
    )
    tick_positions_px = (tick_positions_pc - cumulative_length_pc) / px_to_pc

    fig.set_ticks(
        x_ticks=tick_positions_px,
        x_tick_labels=list(map(round, tick_positions_pc)),
        y_ticks=velocity_tick_positions,
        y_tick_labels=list(map(round, velocity_tick_values))
    )

    cumulative_length_pc += current_aperture_length_pc
    aperture_lengths_px.append(current_aperture_length_pc / px_to_pc)

figs[0].set_tick_params(draw_left_labels=True, draw_left_ticks=True)

pv_fig = gl.SmartFigure(
    num_cols=len(figs),
    x_label="Position along the aperture [pc]",
    y_label=r"Pa$\alpha$ velocity [km s$^{{-1}}$]",
    elements=figs,
    width_padding=0.0028,
    width_ratios=aperture_lengths_px
)

fig = gl.SmartFigure(
    num_cols=2,
    remove_x_ticks=True,
    remove_y_ticks=True,
    aspect_ratio=[1],
    size=(10, 3),
    width_ratios=[1, 2],
    reference_labels=False,
    elements=[[hm, *apertures, AGN_pos], pv_fig],
)

fig.save("figures/letter/pv_diagram.pdf")